In [ ]:
rm service wip changes

In [ ]:
SELECT table_schema, table_name, column_name
FROM information_schema.columns
WHERE LOWER(column_name) = 'cprod_group_conformed'
ORDER BY table_schema, table_name;

In [ ]:
SELECT table_schema, table_name, column_name
FROM information_schema.columns
WHERE LOWER(table_name) LIKE 'silver_tm3%'
  AND (
       LOWER(column_name) = 'cprod_group_conformed'
       OR LOWER(column_name) LIKE '%cprod%'
       OR LOWER(column_name) LIKE '%group_conformed%'
      )
ORDER BY table_name, column_name;

In [ ]:
ffff

In [ ]:
tables = spark.sql("SHOW TABLES IN silver").collect()

matches = []

for row in tables:
    table_name = row["tableName"]
    full_name = f"silver.{table_name}"
    try:
        cols = spark.sql(f"DESCRIBE {full_name}").collect()
        for c in cols:
            col_name = c["col_name"]
            if col_name:
                cname = col_name.strip().lower()
                if "cprod" in cname or "group_conformed" in cname:
                    matches.append((full_name, col_name))
    except Exception:
        pass

display(spark.createDataFrame(matches, ["table_name", "column_name"]))

In [ ]:
Reviewed and the implementation looks correct. For care_epi_is_test, TM3 stock_item_description is mapped to silver_rdm_care_product.cprod_name using cprod_src_sys_inst_id = 'TM3001', and the flag is set based on cprod_group_conformed = 'Test data'. This aligns with the agreed definition.

In [ ]:
For care_epi_is_test, silver_tm3_fact_appointments.stock_item_fk is used to join to silver_tm3_dim_stock_items on appt.stock_item_fk = si.stock_item_pk. From there, the stock_item_description column is taken and mapped to silver_rdm_care_product.cprod_name, with an additional join condition cprod_src_sys_inst_id = 'TM3001'. After that, the logic uses silver_rdm_care_product.cprod_group_conformed. If cprod_group_conformed = 'Test data', then care_epi_is_test is set to 1; otherwise, it is set to 0.


In [ ]:
“contr_name and contr_src_id were simple text fields, so they were mapped directly from the source dataset into the SharePoint list. But contr_src_sys_inst_id was a SharePoint lookup field, so we could not pass the text value directly. Instead, we searched the RDM - Source System Instance reference list for the matching source system instance value, took the SharePoint ID of that matching row, and passed that ID into the target lookup field. That is how SharePoint created the proper relationship and displayed the correct lookup value.”

In [ ]:
A lookup field does not store plain text. It stores the ID of the related row from the reference list. So we first found the matching source system instance row in the master list, took its SharePoint ID, and passed that ID into the target contract list lookup field.

In [ ]:
That field did not just need a value. It needed a valid link to a row in another SharePoint list, and SharePoint creates that link using the row ID.

In [ ]:
So internally, the lookup field stores something like:
“This contract row is linked to row ID 2 in the RDM - Source System Instance list.”

Then SharePoint shows the related display value on the screen:
CF001

In [ ]:
Completed source-to-target mapping for the SharePoint list ingestion in Dataflow Gen2.
Reviewed Monday attribute definitions against the SharePoint list and finalized the in-scope source fields.
Configured the Dataflow to retain only the required columns, expanded lookup/person fields, and renamed columns to align with the target attribute names.

Implemented in Dataflow Gen2:

src_sys_inst_id
src_sys_inst_src_id
src_sys_inst_name
src_sys_src_id
src_sys_id
src_sys_inst_organisation_name
src_sys_inst_organisation_short_name
z_src_is_active
z_src_created_date_time
z_src_created_by_user
z_src_modified_date_time
z_src_modified_by_user

Out of scope and excluded from this task:

src_sys_inst_bu_id
src_sys_inst_mstr_service_id
src_sys_inst_mstr_service_conformed

Note:
z_record_created_by_user, z_record_created_date_time, z_record_modified_by_user, z_record_modified_date_time, and z_record_is_active are not source SharePoint fields. These will need to be handled in downstream UDM/load logic rather than in the source Dataflow.

In [ ]:
xample User Story Description

User Story: Add logic for RDM Contract add table

Description:
Create logic to populate the RDM Contract add table for the required source systems.
The output should include only new rows that are not already present in the existing RDM Contract table.
The required fields should be mapped based on the agreed Monday definitions for each source system.
Systems in scope for this story are WIP, MPB, CF, IAPTUS, and SystemOne.
Any unclear business rule, such as source id uniqueness, should be confirmed with the business owner before finalising the implementation.

Example Task Description

Task: Add code for RDM Contract add table for CF

Description:
Implement the CF-specific logic for the RDM Contract add table.
Map the required output fields from the agreed CF source data based on the Monday definition.
Ensure only rows not already present in the existing RDM Contract table are included.
Validate the output and confirm it matches the expected source mapping and business rules.